### GD_Landsat_02_Download selects Landsat images of SEAN glaciers and downloads them.
TODO: multiple years/multiple satellites/whole record  
See various geemap materials - Courses and Notebooks, as well as Google Developers for Google Earth Engine  
https://geemap.org/notebooks/01_geemap_intro/  
https://developers.google.com/earth-engine/guides/image_overview  
See also: GD_Landsat_01_Setup.ipynb, test_proj, test_roi, gee_intro/AssetManagement/export_data, image_visualization, 50_cartoee_quickstart  
Test outputs are here: C:\Users\andyb\Documents\U\GEE-Courses\data and shown in test_geemap.qgz  
  
set up for Landsat in UTM8N. True for path 059. Path 060 uses UTM7N (grr!).  
Note that each image band's metadata has crs info e.g. 'crs': 'EPSG:32608', 'crs_transform': [30, 0, 289785, 0, -30, 6631515]}  

In [ ]:
import ee
import geemap
import pandas as pd
import os
from shapely.geometry import box, Polygon

In [ ]:
#this or ee.Initialize
Map = geemap.Map()

In [ ]:
glaciers = pd.read_csv(r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics\glacierPropsLandsat.csv')
glaciers['Name']

In [ ]:
#choose one glacier 
glacier = glaciers.iloc[1]
glacierdf=glaciers.iloc[[1]]
print('You chose: ' + glacier['Name'])
out_dir=os.path.join(r'C:\Users\andyb\Documents\U\GEE-Courses\data', glacier['Name']+' Test 3')

In [ ]:
glacierPt = ee.Geometry.Point(glacier['LonCenter'],glacier['LatCenter']) #-137.121, 58.838) #Johns Hopkins (center of terminus)
#TODO: explore polygon instead (may work seamlessly, may not)
#SEE ALSO: reducing/clipping to area in "create an image composite" section of reducing_image_collection.ipynb
Map.centerObject(glacierPt, 12)  # Zoom level 12 for a close view
# Add a marker at the point (optional, for visualization)
Map.addLayer(glacierPt, {'color': 'red'}, glacier['Name'])

In [ ]:
type(glacierdf)

In [ ]:
#add region of interest
#roi = ee.Geometry.Rectangle([glacier['LonMin'], glacier['LatMin'], glacier['LonMax'], glacier['LatMax']])
#roi = ee.Geometry.Rectangle([glacier['LonMinX'], glacier['LatMinY'], glacier['LonMaxX'], glacier['LatMaxY']])
def load_polygon(df):
    # Extract coordinates from the single row
    coords = []
    i = 1
    while f'x{i}' in df.columns and f'y{i}' in df.columns:
        x = df[f'x{i}'].iloc[0]
        y = df[f'y{i}'].iloc[0]
        coords.append((x, y))
        i += 1
    # Ensure the polygon is closed (first and last points are the same)
    if coords and coords[0] != coords[-1]:
        coords.append(coords[0])
    return Polygon(coords)

roi=load_polygon(glacierdf)
#shapely back to ee_polygon: Get the exterior coordinates as a list of lists
roicoords = [list(roi.exterior.coords)]
roi_ee = ee.Geometry.Polygon(roicoords)
print(roi)
print(roi_ee)

In [ ]:
# Create a feature with the region name
feature = ee.Feature(roi_ee, {'name': glacier['Name']+" "+glacier['Region']})
# Add the region to the map with a unique color or style
Map.addLayer(feature, {'color': 'red'}, glacier['Name']+" "+glacier['Region'])

In [ ]:
#see GD_MORW_2024_GEEDiT2_04 for examples of other satellites ~line 400
collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterDate('2014-01-01', '2015-01-01')
    .filterBounds(glacierPt)
    .sort('system:time_start')
)
collectionSR = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterDate('2014-01-01', '2015-01-01')
    .filterBounds(glacierPt)
    .sort('system:time_start')
)
print(collection.size().getInfo())
collection.size()

In [ ]:
collection.aggregate_array("system:id").getInfo()

#CONCLUSION: top of atmosphere has 3 extra images (25 total) compared to surface reflectance (22 total):  
 #'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20141224', #extra
 #'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20141113', #extra
 #'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20141129'] #extra
#100% Cloudy? NO, but low sun angle...

In [ ]:
#geemap metadata about the image (properties in alphabetical order)
first_image = collection.first()
first_imageSR = collectionSR.first()
first_info = geemap.image_props(first_image).getInfo() #DATE_ACQUIRED
first_infoSR = geemap.image_props(first_imageSR).getInfo() #DATE_ACQUIRED
first_info

In [ ]:
first_date=first_info['DATE_ACQUIRED']
first_dateSR=first_infoSR['DATE_ACQUIRED']
first_date+' '+first_dateSR

In [ ]:
#ee.Image.getInfo() to retrieve metadata about the image
image_info = first_image.getInfo()
#NOTE: from grok "Use getInfo() Sparingly: Calling getInfo() forces a synchronous server request, which can be slow and may hit API limits. Use it for debugging or when you need specific metadata in Python.
#Alternative: For large-scale processing, prefer server-side operations (e.g., ee.Image.get() or ee.Image.select()) to avoid fetching data to the client."

image_info #dictionary with sub-dictionary "properties"
#each band has crs info e.g. 'crs': 'EPSG:32608', 'crs_transform': [30, 0, 289785, 0, -30, 6631515]},

In [ ]:
image_infoP=image_info['properties']
image_infoP #also dictionary
#CONCLUSION: similar lists - from geemap vs from ee.

In [ ]:
#first_info vs image_info
print(type(first_info))
print(type(image_infoP))

common_keys = set(first_info.keys()) & set(image_infoP.keys())
print('Common',common_keys)

#in 1 but not 2
diff_keys = set(first_info.keys()) - set(image_infoP.keys())
print('in 1 but not 2',diff_keys)

#in 2 but not 1
diff_keys2 = set(image_infoP.keys()) - set(first_info.keys())
print('in 2 but not 1',diff_keys2)

In [ ]:
# Print selected metadata
print("Image ID:", image_info['id'], first_info['system:id'])
print("Date Acquired:", image_info['properties']['DATE_ACQUIRED'], first_info['DATE_ACQUIRED'])
print("Cloud Cover:", image_info['properties']['CLOUD_COVER'], first_info['CLOUD_COVER'])
print("Band Names:", [band['id'] for band in image_info['bands']])
print("Band Names2:", first_info['system:band_names'])
print("Band NamesSR:", first_infoSR['system:band_names'])

In [ ]:
# Fetch metadata for all images in the collection and create a metadata DataFrame with the info
mdf = pd.DataFrame({
    'system:id': collection.aggregate_array('system:id').getInfo(), # Image IDs (with path)
    'system:index': collection.aggregate_array('system:index').getInfo(), # Image IDs (just image name)
    'DATE_ACQUIRED': collection.aggregate_array('DATE_ACQUIRED').getInfo(), # date acquired
    'system:time_start': collection.aggregate_array('system:time_start').getInfo(), # Acquisition timestamps
    'CLOUD_COVER': collection.aggregate_array('CLOUD_COVER').getInfo(),  # Cloud cover %
    'CLOUD_COVER_LAND': collection.aggregate_array('CLOUD_COVER_LAND').getInfo()  # Cloud cover land %    
})
# Convert timestamps to readable dates (double check that acquired date is same as image start time). Guessing this is UTC.
mdf['datetime'] = pd.to_datetime(mdf['system:time_start'], unit='ms').dt.strftime('%Y-%m-%d %H:%M:%S')
mdf

In [ ]:
# Save metadata to CSV
info_file = os.path.join(out_dir, 'LandsatMetadata.csv')
mdf.to_csv(info_file, index=False)
print(f"Image info saved to: {info_file}")

In [ ]:
# Add the Landsat image to the map (visualize with true color bands) USE FOR SURFACE REFLECTANCE
vis_paramsSR = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # Red, Green, Blue
    'min': 2000, #was 8000
    'max': 150000 #was 18000
}
# Add the Landsat TOA image to the map (visualize with true color bands) TOP OF ATMOSPHERE
#GEEDiT uses bands:['B3','B2','B1'],gamma:1.5,min:0,max:0.8
vis_params = {
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue for TOA
    'min': 0.05,
    'max': 1.6 #was 0.3 or 0. for normal land scenes, not snow
}
Map.addLayer(first_image, vis_params, "First image "+first_date)
Map.addLayer(first_imageSR, vis_paramsSR, "First image SR "+first_date)
Map
#CONCLUSION: SR product seems to have the highlights washed out - details seen in TOA not present in SR, even with max limit set quite high. 
#Would be nice to be more systematic about this, but enough for now...
#Is there a way to plot the histogram of values? Or set min/max to 5%/95%?

## Time Series of Images

In [ ]:
Map.add_time_slider(collection, vis_params, labels=dates, time_interval=1)
Map

## Download
DONE: Where in the download is there reprojection? Nowhere. Google Earth is in WGS84. Landsat single scenes are in UTM8N or 7N for this part of the world. Some Landsat examples (e.g. SF airport) are larger mosaics so they also use WGS84. GEEDiT for glaciers shows the same as what I see here.
Johns Hopkins ones come out as EPSG:32608 UTM8N. 
Other examples come out as WGS84... Proj4: +proj=utm +zone=8 +datum=WGS84 +units=m +no_defs

TODO: Standardize/understand display parameters in jupyter and in QGIS. 
NOTE: parameters vary based on Lansdat satellite.

In [ ]:
# Function to clip each image in the collection to the ROI
def clip_image(image):
    #return image.clip(roi)
    return image.clip(roi_ee)

# Apply clipping to the entire collection
collection_clip = collection.map(clip_image)

In [ ]:
Map.addLayer(collection_clip.first(), vis_params, "First clipped "+first_date)

In [ ]:
#export .tif
geemap.ee_export_image_collection(collection_clip, out_dir=out_dir)

In [ ]:
#Export png
#Convert collection to a list to iterate over images
image_list = collection_clip.toList(collection_clip.size())

# Get collection size
size = collection_clip.size().getInfo()

#Loop through each image in the collection
for i in range(size):
    try:
        # Get the image from the list
        image = ee.Image(image_list.get(i))
        
        # Get image ID for filename
        image_id = image.get('system:id').getInfo()
        
        # Define output filename
        filename = os.path.join(out_dir, f'{image_id.replace("/", "_")}.png') #e.g. USDA/NAIP/DOQQ/m_4609915_sw_14_h_20170703
        
        # Generate thumbnail
        geemap.get_image_thumbnail(image,filename,vis_params=vis_params,dimensions=2000,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test
        print(f'Thumbnail generated for {image_id}: {filename}')
    except Exception as e:
        print(f'Error processing image {i}: {e}')

print('Thumbnail generation complete!')

In [ ]:
print(type(collection_clip.first()))
print(collection_clip.first().get('system:id').getInfo())
print(type(collection_clip.first().get('system:id').getInfo()))
#print(collection_clip.first().get('system:id'))
print(type(collection_clip.first().get('system:id')))

In [ ]:
#FAILED: .map is trying to do things server-side and .getInfo is client-side. Can't mix the two. 
#ASIDE: Not sure why try/catch doesn't operate.
#test if we can do this on the collection instead of reloading each image with ee.Image
def export_thumb(image):
    try:
        # Get image ID for filename
        image_id = image.get('system:id').getInfo()
        # Define output filename
        filename = os.path.join(out_dir, f'c{image_id.replace("/", "_")}.png') #e.g. USDA/NAIP/DOQQ/m_4609915_sw_14_h_20170703
        # Generate thumbnail
        geemap.get_image_thumbnail(image,filename,vis_params=vis_params,dimensions=2000,crs='EPSG:32608') #UTM Zone 8N (EPSG:32608) 3338 Alaska Albers as a test
        print(f'Thumbnail generated for {image_id}: {filename}')
    except Exception as e:
        print(f'Error processing image {i}: {e}')
    return None #try to avoid error "User-defined methods must return a value"

# Apply export_thumb to the entire collection
#FAILS: collection_clip.map(export_thumb)